# 🚀 Sasol Customer Value Recruitment Challenge
## Autonomous Grandmaster Multi-Task Predictive Architecture (Google Colab Edition)

This notebook runs the end-to-end, leak-free competitive ML pipeline for the Sasol Customer Value Recruitment Challenge:
1. **Head 1:** CLV Fuel 3-month volume prediction ($z_{\text{fuel}} = \ln(1 + y_{\text{fuel}}) / s_{\text{fuel}}$)
2. **Head 2:** CLV Non-Fuel 3-month store spend prediction ($z_{\text{nonfuel}} = \ln(1 + y_{\text{nonfuel}}) / s_{\text{nonfuel}}$) via Two-Stage Hurdle
3. **Head 3:** Opportunity Behavioral Transition Multi-Class Classifier conditioned on expected monetary predictions + Nelder-Mead F1 Threshold Optimizer
4. **Hugging Face Semantic Layer:** Spend-weighted customer taste vectors & category semantic cosine affinities

> **Hardware Acceleration:** Compatible with Google Colab GPU (T4/V100/A100) or High-RAM CPU runtimes.

In [ ]:
# 1. Environment & Hardware Diagnostics
import os, sys, platform, subprocess
print(f"OS: {platform.system()} {platform.release()}")
print(f"Python Version: {sys.version}")
try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode()
    print(f"GPU Detected: {gpu_info.strip()}")
except Exception:
    print("No GPU detected (running in CPU mode). CPUs will be multi-threaded.")


In [ ]:
# 2. Install High-Performance Competition Libraries
!pip install -q polars lightgbm catboost xgboost scikit-learn scipy pyyaml torch sentence-transformers

In [ ]:
# 3. Data Ingestion Setup (Google Drive or Zip Upload)
from pathlib import Path
import zipfile

# OPTION A: If you uploaded 'sasol_full_bundle.zip' directly to Colab:
bundle_zip = Path("sasol_full_bundle.zip")
if bundle_zip.exists():
    print(f"Unpacking {bundle_zip}...")
    with zipfile.ZipFile(bundle_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Unpacking complete!")
else:
    print("Bundle zip not found in working dir. Checking for mounted Google Drive...")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Drive mounted successfully.")
    except Exception as e:
        print("Google Drive not mounted or not in Colab.")

# Verify directory structure
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("submissions").mkdir(parents=True, exist_ok=True)
print("Directory structure verified:", [p.name for p in Path('data').iterdir()])


In [ ]:
# 4. Seed Locking & Authoritative Target Constants
import os, random, numpy as np, pandas as pd, polars as pl

SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)

# Authoritative Normalization & Label Metadata from label_config.json
S_FUEL = 2.352825386185113
S_NONFUEL = 2.4200904988927476

OPPORTUNITY_CLASSES = [
    "Existing-category growth: Beverages",
    "Existing-category growth: Food & beverage",
    "Existing-category growth: Fuel",
    "Existing-category growth: Other",
    "Inactivity",
    "New category adoption: Beverages",
    "New category adoption: Braai & Ice",
    "New category adoption: Breads & rolls",
    "New category adoption: Confectionary",
    "New category adoption: Food & beverage",
    "New category adoption: Groceries",
    "New category adoption: Ice-cream",
    "New category adoption: Lubricants",
    "New category adoption: Other",
    "New category adoption: Snacking",
    "New category adoption: Tobacco",
    "Stable"
]
CLASS_MAP = {c: i for i, c in enumerate(OPPORTUNITY_CLASSES)}
INV_CLASS_MAP = {i: c for i, c in enumerate(OPPORTUNITY_CLASSES)}
print(f"Config locked. Opportunity classes: {len(OPPORTUNITY_CLASSES)}")


In [ ]:
# 5. Polars-Powered High-Velocity Feature Factory
from datetime import datetime

def compute_customer_features(transactions_path: str, cutoff_date: str) -> pl.DataFrame:
    cutoff_dt = datetime.strptime(cutoff_date, "%Y-%m-%d")
    df = pl.read_csv(transactions_path, infer_schema_length=0)
    
    df = df.with_columns([
        pl.coalesce([
            pl.col("DateTimeZA").str.strptime(pl.Datetime, format="%Y-%m-%dT%H:%M:%S%.f", strict=False),
            pl.col("DateTimeZA").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False),
            pl.col("DateTimeZA").str.strptime(pl.Datetime, format="%Y-%m-%dT%H:%M:%S%Z", strict=False),
            pl.col("DateTimeZA").str.strptime(pl.Datetime, format="%Y-%m-%d", strict=False),
        ]).alias("dt"),
        pl.col("Quantity").cast(pl.Float64, strict=False).fill_null(0.0).alias("qty"),
        pl.col("TotalExclAmount").cast(pl.Float64, strict=False).fill_null(0.0).alias("amount"),
        pl.col("ItemCategoryLevel2").fill_null("Unknown").alias("cat_l2"),
        pl.col("LineItemType").fill_null("").alias("item_type"),
        pl.col("LocationId").fill_null("Unknown").alias("loc_id"),
        pl.col("TransactionId").fill_null("Unknown").alias("txn_id"),
    ])
    
    df = df.filter(pl.col("dt") < cutoff_dt)
    df = df.with_columns([
        (cutoff_dt - pl.col("dt")).dt.total_days().alias("days_before_cutoff"),
        pl.col("dt").dt.hour().alias("hour"),
        pl.col("dt").dt.weekday().alias("weekday"),
        pl.col("dt").dt.day().alias("day_of_month"),
        pl.col("dt").dt.date().alias("txn_date"),
    ])
    
    is_fuel = (pl.col("cat_l2") == "Fuel") & (pl.col("item_type") == "Sale_Fuel")
    is_nonfuel = (pl.col("cat_l2") != "Fuel") & (~pl.col("cat_l2").is_in(["Value added services", "Print media", "Carrier Bags"]))
    
    df = df.with_columns([
        is_fuel.alias("is_fuel"),
        is_nonfuel.alias("is_nonfuel"),
        pl.when(is_fuel).then(pl.col("qty")).otherwise(0.0).alias("fuel_qty"),
        pl.when(is_fuel).then(pl.col("amount")).otherwise(0.0).alias("fuel_spend"),
        pl.when(is_nonfuel).then(pl.col("amount")).otherwise(0.0).alias("nonfuel_spend"),
    ])
    
    aggs = [
        pl.col("days_before_cutoff").min().alias("recency_days"),
        pl.col("days_before_cutoff").max().alias("tenure_days"),
        pl.col("txn_id").n_unique().alias("lifetime_txn_count"),
        pl.col("amount").sum().alias("lifetime_spend"),
        pl.col("fuel_qty").sum().alias("lifetime_fuel_litres"),
        pl.col("nonfuel_spend").sum().alias("lifetime_nonfuel_spend"),
        pl.col("cat_l2").n_unique().alias("unique_categories_lifetime"),
        pl.col("loc_id").n_unique().alias("unique_stations_lifetime"),
    ]
    
    for w in [14, 30, 60, 90, 180]:
        mask = pl.col("days_before_cutoff") <= w
        aggs.extend([
            pl.col("txn_id").filter(mask).n_unique().alias(f"freq_{w}d"),
            pl.col("amount").filter(mask).sum().alias(f"spend_{w}d"),
            pl.col("fuel_qty").filter(mask).sum().alias(f"fuel_litres_{w}d"),
            pl.col("nonfuel_spend").filter(mask).sum().alias(f"nonfuel_spend_{w}d"),
            pl.col("cat_l2").filter(mask).n_unique().alias(f"unique_cats_{w}d"),
            pl.col("loc_id").filter(mask).n_unique().alias(f"unique_locs_{w}d"),
        ])
        
    aggs.extend([
        pl.col("txn_id").filter((pl.col("hour") >= 6) & (pl.col("hour") <= 9)).n_unique().alias("morning_commute_txns"),
        pl.col("txn_id").filter((pl.col("hour") >= 16) & (pl.col("hour") <= 19)).n_unique().alias("evening_commute_txns"),
        pl.col("txn_id").filter(pl.col("weekday") >= 6).n_unique().alias("weekend_txns"),
        pl.col("txn_id").filter(pl.col("weekday") < 6).n_unique().alias("weekday_txns"),
        pl.col("txn_id").filter((pl.col("day_of_month") >= 25) | (pl.col("day_of_month") <= 2)).n_unique().alias("paycycle_txns"),
        pl.col("amount").filter((pl.col("day_of_month") >= 25) | (pl.col("day_of_month") <= 2)).sum().alias("paycycle_spend"),
    ])
    
    base_agg = df.group_by("ID").agg(aggs)
    
    station_spend = df.group_by(["ID", "loc_id"]).agg(pl.col("amount").sum().alias("station_spend"))
    primary_station = station_spend.sort(["ID", "station_spend"], descending=[False, True]).group_by("ID").first()
    primary_station = primary_station.select([pl.col("ID"), pl.col("station_spend").alias("primary_station_spend")])
    base_agg = base_agg.join(primary_station, on="ID", how="left")
    
    distinct_visits = df.select(["ID", "txn_date"]).unique().sort(["ID", "txn_date"])
    ipi_df = distinct_visits.with_columns([
        (pl.col("txn_date") - pl.col("txn_date").shift(1).over("ID")).dt.total_days().alias("ipi_days")
    ]).filter(pl.col("ipi_days").is_not_null())
    
    ipi_stats = ipi_df.group_by("ID").agg([
        pl.col("ipi_days").mean().alias("ipi_mean"),
        pl.col("ipi_days").std().alias("ipi_std"),
        pl.col("ipi_days").median().alias("ipi_median"),
        pl.col("ipi_days").min().alias("ipi_min"),
        pl.col("ipi_days").max().alias("ipi_max"),
    ])
    base_agg = base_agg.join(ipi_stats, on="ID", how="left")
    
    feats = base_agg.with_columns([
        (pl.col("spend_30d") / ((pl.col("spend_90d") / 3.0) + 1e-4)).alias("spend_velocity_30_90"),
        (pl.col("freq_14d") / ((pl.col("freq_60d") / 4.28) + 1e-4)).alias("visit_velocity_14_60"),
        (pl.col("spend_14d") / ((pl.col("spend_60d") / 4.28) + 1e-4)).alias("spend_velocity_14_60"),
        (pl.col("lifetime_spend") / (pl.col("lifetime_txn_count") + 1e-4)).alias("avg_basket_spend_lifetime"),
        (pl.col("spend_90d") / (pl.col("freq_90d") + 1e-4)).alias("avg_basket_spend_90d"),
        (pl.col("lifetime_nonfuel_spend") / (pl.col("lifetime_spend") + 1e-4)).alias("nonfuel_spend_ratio_lifetime"),
        (pl.col("nonfuel_spend_90d") / (pl.col("spend_90d") + 1e-4)).alias("nonfuel_spend_ratio_90d"),
        (pl.col("primary_station_spend") / (pl.col("lifetime_spend") + 1e-4)).alias("station_loyalty_index"),
        ((pl.col("lifetime_fuel_litres") > 0) & (pl.col("lifetime_nonfuel_spend") <= 0)).cast(pl.Int32).alias("flag_fuel_only"),
        ((pl.col("lifetime_fuel_litres") <= 0) & (pl.col("lifetime_nonfuel_spend") > 0)).cast(pl.Int32).alias("flag_store_only"),
        ((pl.col("lifetime_fuel_litres") > 0) & (pl.col("lifetime_nonfuel_spend") > 0)).cast(pl.Int32).alias("flag_dual_shopper"),
        (pl.col("weekend_txns") / (pl.col("lifetime_txn_count") + 1e-4)).alias("weekend_txn_ratio"),
        ((pl.col("morning_commute_txns") + pl.col("evening_commute_txns")) / (pl.col("lifetime_txn_count") + 1e-4)).alias("commute_rush_ratio"),
        ((pl.col("spend_30d") - (pl.col("spend_90d") - pl.col("spend_30d")) / 2.0)).alias("spend_acceleration_delta"),
    ])
    return feats

print("Polars feature factory defined successfully.")


In [ ]:
# 5B. Hugging Face Semantic Feature Extractor (Customer Taste & Category Affinity Vectors)
from sklearn.decomposition import TruncatedSVD
import torch

OPPORTUNITY_CATEGORY_DESCRIPTIONS = {
    'Beverages': 'Beverages sodas cold drinks fruit juice energy drinks water iced tea',
    'Braai & Ice': 'Braai charcoal briquettes firewood firelighters bag of ice ice cubes',
    'Breads & rolls': 'Fresh bakery bread white brown rolls buns loaves baguettes',
    'Confectionary': 'Confectionary chocolates sweets candy gums gummies mints chocolate bars',
    'Food & beverage': 'Food and beverage prepared meals pies toasted sandwiches hot food takeaway lunch',
    'Fuel': 'Fuel service station diesel turbo petrol unleaded 93 95 pump litres fill up',
    'Groceries': 'Groceries pantry items milk sugar coffee tea flour cooking oil tinned food',
    'Ice-cream': 'Ice cream frozen dairy treats cones popsicles ice lollies tubs',
    'Lubricants': 'Motor lubricants engine oil motor oil 5W30 10W40 brake fluid coolant transmission',
    'Snacking': 'Snacks potato crisps chips biltong droewors dried fruit roasted nuts peanuts pretzels',
    'Tobacco': 'Tobacco cigarettes cigars smoking lighters matches vape rolling paper',
    'Automotive Products': 'Automotive car care wiper blades fuses tyre shine screen wash polish',
    'Car Wash': 'Car wash vehicle cleaning vacuum wash and dry exterior auto detailing',
    'Coffee': 'Fresh brewed coffee espresso cappuccino latte americano hot coffee beverage',
    'Frozen food': 'Frozen meals frozen vegetables burger patties frozen chicken fish',
    'Perishables': 'Perishable goods fresh dairy yogurt cheese butter eggs',
    'Other': 'General miscellaneous store goods convenience merchandise',
}

def extract_customer_semantic_features(raw_transactions_path: str, cutoff_date: str, model_name: str = 'sentence-transformers/all-MiniLM-L6-v2', n_components: int = 12, seed: int = 42) -> pl.DataFrame:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        return pl.DataFrame({'ID': []})

    cutoff_dt = datetime.strptime(cutoff_date, '%Y-%m-%d')
    df = pl.read_csv(raw_transactions_path, infer_schema_length=0)
    df = df.with_columns([
        pl.coalesce([
            pl.col('DateTimeZA').str.strptime(pl.Datetime, format='%Y-%m-%dT%H:%M:%S%.f', strict=False),
            pl.col('DateTimeZA').str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S', strict=False),
            pl.col('DateTimeZA').str.strptime(pl.Datetime, format='%Y-%m-%dT%H:%M:%S%Z', strict=False),
            pl.col('DateTimeZA').str.strptime(pl.Datetime, format='%Y-%m-%d', strict=False),
        ]).alias('dt'),
        pl.col('TotalExclAmount').cast(pl.Float64, strict=False).fill_null(0.0).alias('amount'),
        pl.col('ItemName').fill_null('Unknown').alias('item_name'),
    ])
    hist_tx = df.filter(pl.col('dt') < cutoff_dt)
    unique_items = hist_tx.select('item_name').filter(pl.col('item_name').is_not_null() & (pl.col('item_name') != 'Unknown')).unique().to_series().to_list()
    if not unique_items:
        return pl.DataFrame({'ID': []})

    cleaned_items = [str(x).strip().upper() for x in unique_items]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SentenceTransformer(model_name, device=device)
    item_embs_384 = model.encode(cleaned_items, batch_size=128, show_progress_bar=False, normalize_embeddings=True)
    actual_comp = min(n_components, len(cleaned_items) - 1, 384)
    svd = TruncatedSVD(n_components=actual_comp, random_state=seed)
    item_embs_red = svd.fit_transform(item_embs_384)
    emb_cols = [f'sem_dim_{i}' for i in range(actual_comp)]
    item_dict = {'item_name': unique_items}
    for i, col_name in enumerate(emb_cols):
        item_dict[col_name] = item_embs_red[:, i].astype(np.float32)
    item_emb_df = pl.DataFrame(item_dict)
    joined = hist_tx.join(item_emb_df, on='item_name', how='inner').with_columns([
        pl.col('amount').clip(lower_bound=0.0).alias('pos_spend')
    ])
    aggs = [pl.col('pos_spend').sum().alias('total_pos_spend')]
    for col_name in emb_cols:
        aggs.append((pl.col(col_name) * pl.col('pos_spend')).sum().alias(f'{col_name}_sum'))
    cust_emb = joined.group_by('ID').agg(aggs)
    norm_exprs = [
        (pl.col(f'{col_name}_sum') / (pl.col('total_pos_spend') + 1e-4)).alias(f'hf_taste_{col_name}')
        for col_name in emb_cols
    ]
    cat_keys = list(OPPORTUNITY_CATEGORY_DESCRIPTIONS.keys())
    cat_descs = [OPPORTUNITY_CATEGORY_DESCRIPTIONS[k] for k in cat_keys]
    cat_embs_384 = model.encode(cat_descs, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    cat_embs_red = svd.transform(cat_embs_384)
    aff_exprs = []
    for cat_idx, cat_name in enumerate(cat_keys):
        cat_vec = cat_embs_red[cat_idx]
        norm_cat = np.linalg.norm(cat_vec) + 1e-5
        cat_unit = cat_vec / norm_cat
        dot_terms = [pl.col(f'hf_taste_{emb_cols[i]}') * float(cat_unit[i]) for i in range(actual_comp)]
        safe_cat_name = cat_name.replace(' ', '_').replace('&', 'and')
        aff_exprs.append(pl.sum_horizontal(dot_terms).alias(f'hf_aff_{safe_cat_name}'))
    return cust_emb.select([pl.col('ID')] + norm_exprs).with_columns(aff_exprs)

print('Hugging Face Semantic Extractor initialized successfully.')


In [ ]:
# 6. Multi-Cohort Target Extraction & Feature Processing
import subprocess

def get_labels(cutoff: str):
    safe = cutoff.replace('-', '_')
    out_p = Path(f'data/processed/labels_{safe}.csv')
    if not out_p.exists():
        print(f"Generating labels for {cutoff} via label_rules.py...")
        cmd = [
            'python', 'data/raw/label_rules.py',
            '--transactions', 'data/raw/train.csv',
            '--cutoff', cutoff,
            '--config', 'data/raw/label_config.json',
            '--output', str(out_p)
        ]
        subprocess.run(cmd, check=True)
    return pd.read_csv(out_p)

def get_features(cutoff: str):
    safe = cutoff.replace('-', '_')
    out_p = Path(f'data/processed/features_{safe}.csv')
    if not out_p.exists():
        print(f"Generating Polars + Hugging Face features for {cutoff}...")
        f_pl = compute_customer_features('data/raw/train.csv', cutoff)
        try:
            sem_pl = extract_customer_semantic_features('data/raw/train.csv', cutoff)
            if sem_pl.width > 1:
                f_pl = f_pl.join(sem_pl, on='ID', how='left')
        except Exception as e:
            print(f"Semantic extraction fallback: {e}")
        df = f_pl.to_pandas()
        df.to_csv(out_p, index=False)
    return pd.read_csv(out_p)

# Pool historical cutoffs for maximum training power
TRAIN_CUTOFFS = ['2024-12-01', '2025-03-01', '2025-06-01']
VAL_CUTOFF = '2025-09-01'
TEST_CUTOFF = '2025-12-01'

train_dfs = []
for c in TRAIN_CUTOFFS:
    f = get_features(c)
    l = get_labels(c)
    m = pd.merge(f, l, on='ID', how='inner')
    m['cohort'] = c
    train_dfs.append(m)

train_df = pd.concat(train_dfs, ignore_index=True)
print(f"Total Multi-Cohort Training Pool: {train_df.shape}")

# Validation set (Strict Out-of-Time)
val_f = get_features(VAL_CUTOFF)
val_l = get_labels(VAL_CUTOFF)
val_df = pd.merge(val_f, val_l, on='ID', how='inner')
print(f"Validation Pool: {val_df.shape}")

# Test set
test_f = get_features(TEST_CUTOFF)
sample_sub = pd.read_csv('data/raw/SampleSubmission.csv')
test_df = pd.merge(sample_sub[['ID']], test_f, on='ID', how='left')
print(f"Test Pool: {test_df.shape}")


In [ ]:
# 7. Decoupled Multi-Head Architecture with Model Ensembling
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
from sklearn.metrics import mean_squared_error, f1_score
from scipy.optimize import minimize

meta_cols = [
    'ID', 'fuel_litres', 'nonfuel_rand', 'Opportunity',
    'winning_source_category', 'winning_increment_rand',
    'CLV_fuel', 'CLV_nonfuel', 'cohort'
]
feature_cols = [c for c in train_df.columns if c not in meta_cols and not c.startswith('_')]
print(f"Total features engineered: {len(feature_cols)}")

X_train = train_df[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
X_val = val_df[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
X_test = test_df[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)

y_fuel_train = train_df['CLV_fuel'].values
y_fuel_val = val_df['CLV_fuel'].values

y_nonfuel_train = train_df['CLV_nonfuel'].values
y_nonfuel_val = val_df['CLV_nonfuel'].values

y_opp_train = train_df['Opportunity'].map(CLASS_MAP).values
y_opp_val = val_df['Opportunity'].map(CLASS_MAP).values

# --- HEAD 1: FUEL CLV ENSEMBLE ---
print("Training Head 1 (Fuel CLV)... ")
lgb_f = lgb.LGBMRegressor(n_estimators=350, learning_rate=0.03, random_state=42, verbose=-1)
lgb_f.fit(X_train, y_fuel_train)
cat_f = cb.CatBoostRegressor(iterations=350, learning_rate=0.03, random_seed=42, verbose=0)
cat_f.fit(X_train, y_fuel_train)
xgb_f = xgb.XGBRegressor(n_estimators=300, learning_rate=0.03, random_state=42, tree_method='hist')
xgb_f.fit(X_train, y_fuel_train)

p_fuel_val = 0.5 * lgb_f.predict(X_val) + 0.3 * cat_f.predict(X_val) + 0.2 * xgb_f.predict(X_val)
p_fuel_test = 0.5 * lgb_f.predict(X_test) + 0.3 * cat_f.predict(X_test) + 0.2 * xgb_f.predict(X_test)
p_fuel_tr = 0.5 * lgb_f.predict(X_train) + 0.3 * cat_f.predict(X_train) + 0.2 * xgb_f.predict(X_train)
p_fuel_val = np.clip(p_fuel_val, 0, None)
p_fuel_test = np.clip(p_fuel_test, 0, None)
p_fuel_tr = np.clip(p_fuel_tr, 0, None)

# --- HEAD 2: NON-FUEL TWO-STAGE HURDLE ---
print("Training Head 2 (Non-Fuel Hurdle)... ")
hurdle_clf = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.03, random_state=42, verbose=-1)
hurdle_clf.fit(X_train, (y_nonfuel_train > 0).astype(int))
pos_idx = (y_nonfuel_train > 0)
hurdle_reg = cb.CatBoostRegressor(iterations=300, learning_rate=0.03, random_seed=42, verbose=0)
hurdle_reg.fit(X_train[pos_idx], y_nonfuel_train[pos_idx])

p_nonfuel_val = hurdle_clf.predict_proba(X_val)[:, 1] * np.clip(hurdle_reg.predict(X_val), 0, None)
p_nonfuel_test = hurdle_clf.predict_proba(X_test)[:, 1] * np.clip(hurdle_reg.predict(X_test), 0, None)
p_nonfuel_tr = hurdle_clf.predict_proba(X_train)[:, 1] * np.clip(hurdle_reg.predict(X_train), 0, None)
p_nonfuel_val = np.clip(p_nonfuel_val, 0, None)
p_nonfuel_test = np.clip(p_nonfuel_test, 0, None)
p_nonfuel_tr = np.clip(p_nonfuel_tr, 0, None)

# --- HEAD 3: OPPORTUNITY MULTI-CLASS (CROSS-TASK CONDITIONED) ---
print("Training Head 3 (Opportunity Multi-Class)... ")
X_tr_opp = X_train.copy()
X_tr_opp['pred_fuel'] = p_fuel_tr
X_tr_opp['pred_nonfuel'] = p_nonfuel_tr

X_val_opp = X_val.copy()
X_val_opp['pred_fuel'] = p_fuel_val
X_val_opp['pred_nonfuel'] = p_nonfuel_val

X_test_opp = X_test.copy()
X_test_opp['pred_fuel'] = p_fuel_test
X_test_opp['pred_nonfuel'] = p_nonfuel_test

lgb_opp = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.04, random_state=42, class_weight='balanced', verbose=-1)
lgb_opp.fit(X_tr_opp, y_opp_train)
cat_opp = cb.CatBoostClassifier(iterations=300, learning_rate=0.04, random_seed=42, auto_class_weights='Balanced', verbose=0)
cat_opp.fit(X_tr_opp, y_opp_train)

probs_val = 0.6 * lgb_opp.predict_proba(X_val_opp) + 0.4 * cat_opp.predict_proba(X_val_opp)
probs_test = 0.6 * lgb_opp.predict_proba(X_test_opp) + 0.4 * cat_opp.predict_proba(X_test_opp)

# --- POST-PROCESSING: NELDER-MEAD THRESHOLD OPTIMIZER ---
print("Optimizing Multi-Class Decision Thresholds for Weighted F1...")
def objective(weights):
    scaled_p = probs_val * weights
    preds = np.argmax(scaled_p, axis=1)
    return -f1_score(y_opp_val, preds, average='weighted')

res = minimize(objective, np.ones(probs_val.shape[1]), method='Nelder-Mead')
best_w = res.x
p_opp_val = np.argmax(probs_val * best_w, axis=1)
p_opp_test = np.argmax(probs_test * best_w, axis=1)
print("Model training and threshold calibration complete!")


In [ ]:
# 8. Out-Of-Time Validation Score Breakdown
rmse_fuel = np.sqrt(mean_squared_error(y_fuel_val, p_fuel_val))
rmse_nonfuel = np.sqrt(mean_squared_error(y_nonfuel_val, p_nonfuel_val))
f1_opp = f1_score(y_opp_val, p_opp_val, average='weighted')
composite_loss = (rmse_fuel + rmse_nonfuel) + (1.0 - f1_opp)
composite_raw = (rmse_fuel + rmse_nonfuel) - f1_opp

print("=" * 50)
print("OUT-OF-TIME VALIDATION PERFORMANCE (2025-09 SNAPSHOT):")
print(f"  CLV Fuel RMSE:                 {rmse_fuel:.4f}")
print(f"  CLV Non-Fuel RMSE:             {rmse_nonfuel:.4f}")
print(f"  Opportunity Weighted F1:       {f1_opp:.4f}")
print(f"  COMPOSITE LOSS (LOWER=BETTER): {composite_loss:.4f}")
print(f"  LEGACY COMPOSITE RAW:          {composite_raw:.4f}")
print("=" * 50)


In [ ]:
# 9. Strict Submission Contract Verification & Download
sub_df = pd.DataFrame({
    "ID": test_df["ID"],
    "CLV_fuel": p_fuel_test,
    "CLV_nonfuel": p_nonfuel_test,
    "Opportunity": [INV_CLASS_MAP[i] for i in p_opp_test]
})

# Automated Sanity Assertions
assert sub_df.shape == sample_sub.shape, f"Shape mismatch! {sub_df.shape} vs {sample_sub.shape}"
assert list(sub_df.columns) == list(sample_sub.columns), "Column names mismatch!"
assert sub_df.isna().sum().sum() == 0, "Submission contains NaNs!"
assert (sub_df["CLV_fuel"] >= 0).all(), "CLV_fuel contains negative values!"
assert (sub_df["CLV_nonfuel"] >= 0).all(), "CLV_nonfuel contains negative values!"
assert set(sub_df["Opportunity"].unique()).issubset(set(OPPORTUNITY_CLASSES)), "Illegal Opportunity classes!"

sub_filename = "submissions/submission_colab_ensemble.csv"
sub_df.to_csv(sub_filename, index=False)
print(f"✅ All assertions passed! Submission saved to {sub_filename}")
print(sub_df.head(10))

# Optional direct download in Colab browser
try:
    from google.colab import files
    files.download(sub_filename)
    print("Triggered browser download for submission.")
except Exception:
    pass
